## Feature Ablation Experiment

### Objective
To determine whether highly correlated features provide
additional predictive information.

We compare different feature subsets using five-fold
stratified cross-validation.

Features will not be removed solely because they are
highly correlated.

In [4]:
import pandas as pd

# Load training data
train_df = pd.read_csv("rice_train.csv")

# Separate input features and target
X_train = train_df.drop(columns=["Class"])
y_train = train_df["Class"]

# Create four feature sets
feature_sets = {
    "All 7 Features": X_train.columns.tolist(),

    "Without Area": [
        col for col in X_train.columns
        if col != "Area"
    ],

    "Without Convex_Area": [
        col for col in X_train.columns
        if col != "Convex_Area"
    ],

    "Without Both": [
        col for col in X_train.columns
        if col not in ["Area", "Convex_Area"]
    ]
}

# Display the feature sets
for name, features in feature_sets.items():
    print(f"{name}: {len(features)} features")
    print(features)
    print("-" * 40)

All 7 Features: 7 features
['Area', 'Perimeter', 'Major_Axis_Length', 'Minor_Axis_Length', 'Eccentricity', 'Convex_Area', 'Extent']
----------------------------------------
Without Area: 6 features
['Perimeter', 'Major_Axis_Length', 'Minor_Axis_Length', 'Eccentricity', 'Convex_Area', 'Extent']
----------------------------------------
Without Convex_Area: 6 features
['Area', 'Perimeter', 'Major_Axis_Length', 'Minor_Axis_Length', 'Eccentricity', 'Extent']
----------------------------------------
Without Both: 5 features
['Perimeter', 'Major_Axis_Length', 'Minor_Axis_Length', 'Eccentricity', 'Extent']
----------------------------------------


In [5]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd

# Five-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for name, features in feature_sets.items():

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ])

    scores = cross_validate(
        model,
        X_train[features],
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "f1_macro": "f1_macro",
            "balanced_accuracy": "balanced_accuracy"
        }
    )

    results.append({
        "Feature Set": name,
        "Number of Features": len(features),
        "Accuracy": scores["test_accuracy"].mean(),
        "Macro F1": scores["test_f1_macro"].mean(),
        "Balanced Accuracy": scores["test_balanced_accuracy"].mean(),
        "Accuracy SD": scores["test_accuracy"].std()
    })

feature_comparison_df = pd.DataFrame(results)

display(feature_comparison_df.round(4))

,Feature Set,Number of Features,Accuracy,Macro F1,Balanced Accuracy,Accuracy SD
0,All 7 Features,7,0.9331,0.9317,0.9317,0.0097
1,Without Area,6,0.9334,0.9320,0.9320,0.0091
2,Without Convex_Area,6,0.9314,0.9300,0.9301,0.0088
3,Without Both,5,0.9318,0.9303,0.9304,0.0084


### Feature Ablation Results and Interpretation

Four feature subsets were evaluated using Logistic Regression
and five-fold stratified cross-validation on the training data.

The model using all seven features achieved 93.31% accuracy
and 93.17% macro F1.

Removing Area produced 93.34% accuracy and 93.20% macro F1,
representing a negligible improvement of 0.03 percentage points
in accuracy.

Removing Convex_Area reduced accuracy to 93.14%, while removing
both Area and Convex_Area produced 93.18% accuracy.

All feature subsets performed similarly. These findings suggest
that Area and Convex_Area contain overlapping predictive
information, consistent with their strong positive correlation.

Decision:
Removing Area is a reasonable six-feature alternative because
it maintains predictive performance with fewer inputs. However,
the differences are too small to establish a clear superiority.

The existing final two-feature Logistic Regression model remains
unchanged.

Limitation:
These are exploratory cross-validation results. The observed
differences have not been established as statistically significant.

In [6]:
# Define the three feature combinations
selected_feature_sets = {
    "Perimeter Only": ["Perimeter"],
    "Major Axis Only": ["Major_Axis_Length"],
    "Both Features": ["Perimeter", "Major_Axis_Length"]
}

selected_results = []

for name, features in selected_feature_sets.items():

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ])

    scores = cross_validate(
        model,
        X_train[features],
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "f1_macro": "f1_macro",
            "balanced_accuracy": "balanced_accuracy"
        }
    )

    selected_results.append({
        "Feature Set": name,
        "Number of Features": len(features),
        "Accuracy": scores["test_accuracy"].mean(),
        "Macro F1": scores["test_f1_macro"].mean(),
        "Balanced Accuracy": scores["test_balanced_accuracy"].mean()
    })

selected_comparison_df = pd.DataFrame(selected_results)

display(selected_comparison_df.round(4))

,Feature Set,Number of Features,Accuracy,Macro F1,Balanced Accuracy
0,Perimeter Only,1,0.9163,0.9144,0.9134
1,Major Axis Only,1,0.9301,0.9287,0.9288
2,Both Features,2,0.9337,0.9323,0.9319


### Selected Feature Comparison: Interpretation

Three Logistic Regression models were compared using
five-fold stratified cross-validation on the training dataset.

- Perimeter only: 91.63% accuracy, 91.44% macro F1.
- Major_Axis_Length only: 93.01% accuracy, 92.87% macro F1.
- Both features: 93.37% accuracy, 93.23% macro F1.

Major_Axis_Length performed better than Perimeter when
each feature was evaluated individually.

Using both features improved accuracy by 0.36 percentage
points compared with Major_Axis_Length alone.

Although the two features are strongly correlated,
the results indicate that combining them provides a small
additional predictive benefit.

Decision:
Retain Perimeter and Major_Axis_Length in the existing
two-feature Logistic Regression model.

Limitation:
The observed improvement is small and has not been
established as statistically significant. Feature selection
and evaluation using the same cross-validation procedure
may also produce optimistic estimates.

The independent test dataset remains untouched by
these supplementary experiments.